# Bays (2014) Figure 3 — Cued vs Uncued + Optimal Weighting

Panels: **a** error distributions (all/cued/uncued), **b** variance, **c** kurtosis, **d** optimal weighting factor.

Extends Figure 2 with **differential weighting**: the cued location gets weight $\\alpha_{\\text{cued}} > 1$ in both encoding and decoding.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = str(Path.cwd().parents[1])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import time
from scipy.special import logsumexp

from core.encoder.gaussian_process import (
    generate_neuron_population, periodic_rbf_kernel, sample_gp_function,
)
from core.encoder.divisive_normalization import dn_pointwise
from core.encoder.poisson_spike import generate_spikes

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Single-location full Poisson ML decoder + circular stats
# (Not in core — core has the multi-item factorised decoder)

def compute_log_likelihood(counts, g, T_d):
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    return 1.0 - np.abs(np.mean(np.exp(1j * errors)))

def circular_kurtosis(errors):
    V = circular_variance(errors)
    rho2 = np.abs(np.mean(np.exp(2j * errors)))
    kappa2 = 1.0 - rho2
    return kappa2 / max(V**2, 1e-15) if V > 1e-10 else 0.0

def compute_deviation_from_normal(errors, n_bins=50):
    from scipy.stats import vonmises
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    emp, _ = np.histogram(errors, bins=bin_edges, density=True)
    V = circular_variance(errors)
    kappa_fit = max(0.01, 1.0 / V - 1) if V > 0.01 else 100.0
    vm_pdf = vonmises.pdf(centers, kappa_fit)
    return {'bin_centers': centers, 'empirical': emp, 'normal_fit': vm_pdf,
            'deviation': emp - vm_pdf}

# Multi-item factorised decoder (Eqs. 23-26)
def compute_spike_weighted_log_tuning(counts, f_list):
    return [counts @ f_k for f_k in f_list]

def compute_marginal_log_likelihood_efficient(L_list, cued_idx):
    ll = L_list[cued_idx].copy()
    for k in range(len(L_list)):
        if k != cued_idx:
            ll = ll + logsumexp(L_list[k])
    return ll

In [ ]:
def generate_population(M, n_theta, lengthscale, n_locations=1, seed=42):
    population = generate_neuron_population(
        n_neurons=M, n_orientations=n_theta, n_locations=n_locations,
        base_lengthscale=lengthscale, lengthscale_variability=0.0, seed=seed)
    thetas = population[0]['orientations']
    f_all = []
    for loc in range(n_locations):
        f_loc = np.array([population[n]['f_samples'][loc, :] for n in range(M)])
        f_all.append(f_loc)
    return thetas, f_all

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
M = 100; N_THETA = 64; N_TRIALS = 5000; N_TRIALS_SWEEP = 1000
T_D = 0.1; SIGMA_SQ = 1e-6; LAMBDA_BASE = 0.5; GAMMA = 100.0
SET_SIZES = [2, 4, 8]; CUE_RATIO = 3.0
ALPHA_RANGE = (1.0, 8.0); N_ALPHA = 12
SEED = 42; N_SEEDS = 2; N_BINS = 50

## Weighted encoder/decoder

Encoding: $\\log r_i^{\\text{pre}} = \\sum_k \\alpha_k \\cdot f_{i,k}(\\theta_k)$

Decoding: $\\ell_{\\text{marginal}}(\\theta_p) = \\alpha_p \\cdot L_p(\\theta_p) + \\sum_{k \\neq p} \\text{logsumexp}(\\alpha_k \\cdot L_k)$

In [ ]:
def compute_weighted_marginal(L_list, alpha_weights, probed_loc):
    ll = alpha_weights[probed_loc] * L_list[probed_loc]
    for k in range(len(L_list)):
        if k != probed_loc:
            ll = ll + logsumexp(alpha_weights[k] * L_list[k])
    return ll

def run_weighted_trials(f_all, thetas, active_locs, cued_loc, probed_loc,
                         alpha_cued, gamma, T_d, sigma_sq, n_trials, rng):
    l = len(active_locs); M, n_theta = f_all[0].shape
    f_active = [f_all[loc] for loc in active_locs]
    alpha_w = np.ones(l); alpha_w[cued_loc] = alpha_cued
    errors = np.empty(n_trials)
    for t in range(n_trials):
        theta_idx = rng.randint(n_theta, size=l)
        log_r = np.zeros(M)
        for k in range(l):
            log_r += alpha_w[k] * f_active[k][:, theta_idx[k]]
        r_pre = np.exp(log_r - np.max(log_r))
        rates = dn_pointwise(r_pre, gamma, sigma_sq)
        counts = generate_spikes(rates, T_d, rng)
        L_list = compute_spike_weighted_log_tuning(counts, f_active)
        ll_m = compute_weighted_marginal(L_list, alpha_w, probed_loc)
        errors[t] = compute_circular_error(thetas[theta_idx[probed_loc]], thetas[np.argmax(ll_m)])
    return errors

## Find optimal alpha + run full trials

In [ ]:
t0 = time.time()
alpha_values = np.linspace(ALPHA_RANGE[0], ALPHA_RANGE[1], N_ALPHA)
max_locs = max(SET_SIZES)
thetas, f_all = generate_population(M, N_THETA, LAMBDA_BASE, max_locs, SEED)

# Phase 1: find optimal alpha per set size
optimal_alphas = {}
for N in SET_SIZES:
    p_c = CUE_RATIO / (CUE_RATIO + N - 1)
    best_cost, best_alpha = np.inf, 1.0
    for alpha in alpha_values:
        rng_c = np.random.RandomState(SEED + N + int(alpha*100))
        ec = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, 0,
                                  alpha, GAMMA, T_D, SIGMA_SQ, N_TRIALS_SWEEP, rng_c)
        rng_u = np.random.RandomState(SEED + N + int(alpha*100) + 50000)
        eu = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, min(1,N-1),
                                  alpha, GAMMA, T_D, SIGMA_SQ, N_TRIALS_SWEEP, rng_u)
        cost = p_c * circular_variance(ec) + (1-p_c) * circular_variance(eu)
        if cost < best_cost:
            best_cost, best_alpha = cost, alpha
    optimal_alphas[N] = best_alpha
    print(f"  N={N}: alpha*={best_alpha:.2f}")

# Phase 2: full trials at optimal alpha
summary = {}
for N in SET_SIZES:
    alpha_opt = optimal_alphas[N]
    rng_c = np.random.RandomState(SEED + N*1000)
    ec = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, 0,
                              alpha_opt, GAMMA, T_D, SIGMA_SQ, N_TRIALS, rng_c)
    rng_u = np.random.RandomState(SEED + N*1000 + 50000)
    eu = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, min(1,N-1),
                              alpha_opt, GAMMA, T_D, SIGMA_SQ, N_TRIALS, rng_u)
    p_c = CUE_RATIO / (CUE_RATIO + N - 1)
    nc = int(round(p_c * N_TRIALS))
    ea = np.concatenate([ec[:nc], eu[:N_TRIALS-nc]])
    summary[N] = {
        'alpha': alpha_opt,
        'vc': circular_variance(ec), 'vu': circular_variance(eu), 'va': circular_variance(ea),
        'kc': circular_kurtosis(ec), 'ku': circular_kurtosis(eu),
        'hc': compute_deviation_from_normal(ec, N_BINS),
        'hu': compute_deviation_from_normal(eu, N_BINS),
        'ha': compute_deviation_from_normal(ea, N_BINS),
    }
    print(f"  N={N}: vc={summary[N]['vc']:.3f} vu={summary[N]['vu']:.3f}")
bins = summary[SET_SIZES[0]]['ha']['bin_centers']
print(f"Done in {time.time()-t0:.1f}s")

## Plot: panels a-d

In [ ]:
RED, GREEN, BLACK, BLUE = '#CC2222', '#228B22', '#222222', '#2255AA'
fig = plt.figure(figsize=(16, 10))
outer = gridspec.GridSpec(1, 2, width_ratios=[2.8, 1], wspace=0.35)
gs_a = gridspec.GridSpecFromSubplotSpec(3, len(SET_SIZES), subplot_spec=outer[0], hspace=0.4, wspace=0.3)
gs_bcd = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer[1], hspace=0.5)
ns = np.array(SET_SIZES, dtype=float)

for row, (key, color, label) in enumerate([('ha',BLACK,'all'), ('hc',RED,'cued'), ('hu',GREEN,'uncued')]):
    for col, N in enumerate(SET_SIZES):
        ax = fig.add_subplot(gs_a[row, col])
        ax.plot(bins, summary[N][key]['empirical'], color=color, lw=1.5)
        ax.set_xlim(-np.pi, np.pi)
        ax.set_xticks([-np.pi,0,np.pi]); ax.set_xticklabels([r'$-\pi$','0',r'$\pi$'])
        if row==0: ax.set_title(f'{N}', fontweight='bold')
        if col==0: ax.set_ylabel('density')

ax_b = fig.add_subplot(gs_bcd[0])
for key,color,label in [('vc',RED,'cued'),('vu',GREEN,'uncued')]:
    ax_b.plot(ns, [summary[N][key] for N in SET_SIZES], 'o-', color=color, lw=1.5, label=label)
ax_b.set_xticks(SET_SIZES); ax_b.set_xlabel('items'); ax_b.set_ylabel('variance'); ax_b.legend(fontsize=8)

ax_c = fig.add_subplot(gs_bcd[1])
for key,color,label in [('kc',RED,'cued'),('ku',GREEN,'uncued')]:
    ax_c.plot(ns, [summary[N][key] for N in SET_SIZES], 'o-', color=color, lw=1.5, label=label)
ax_c.set_xticks(SET_SIZES); ax_c.set_xlabel('items'); ax_c.set_ylabel('kurtosis'); ax_c.legend(fontsize=8)

ax_d = fig.add_subplot(gs_bcd[2])
ax_d.bar(range(len(SET_SIZES)), [summary[N]['alpha'] for N in SET_SIZES], 0.5, color=BLUE, alpha=0.85)
ax_d.axhline(1, color='gray', ls='--', lw=0.8)
ax_d.set_xticks(range(len(SET_SIZES))); ax_d.set_xticklabels([str(n) for n in SET_SIZES])
ax_d.set_xlabel('items'); ax_d.set_ylabel(r'$\alpha_{cued}$')

fig.suptitle('Bays (2014) Fig 3 — Cued vs Uncued', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()